# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive exploration of the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://mlcommons.org/croissant/) library. We'll demonstrate how to discover, extract, and analyze records based on their Croissant schema structure.

### Dataset Source
This dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and print basic info
dataset = mlc.Dataset(croissant_url)
ds_metadata = dataset.metadata

print(f"Dataset: {ds_metadata.name}\n\nDescription: {ds_metadata.description}")

## 2. Data Overview
Review available record sets and their fields, referencing all entities by their `@id`s.

In [ ]:
# Explore all record sets (tables) and list their fields and columns.

record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

overview_info = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    field_ids = []
    print("  Fields and columns:")
    for field in rs.fields:
        col_ids = [col.id for col in (field.columns or [])]
        print(f"   - Field: {field.name}\n     @id: {field.id}\n     Columns: {col_ids}")
        field_ids.append(field.id)
    overview_info.append({"record_set_id": rs.id, "field_ids": field_ids})
    print("")

## 3. Data Extraction
Extract all data records from one or more record sets and load them into DataFrames for downstream analysis. Make sure to reference the record set and field `@id`s from the previous overview.

In [ ]:
# Collect all RecordSet @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Create DataFrames for each record set
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[rs_id] = df
    print(f"@id: {rs_id} - Loaded {len(df)} records. Columns: {list(df.columns)}")
    print(df.head(2))
    print("---\n")

## 4. Exploratory Data Analysis (EDA)
Apply data cleaning and manipulation steps. We'll select a numeric field for filtering, normalization, and group-by procedures. Make sure to reference fields by their `@id`s.

In [ ]:
# For demonstration, we use the first non-empty RecordSet and its numeric fields.
import numpy as np

# Helper: Find first dataframe with data
selected_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rsid
        break

if selected_record_set_id is None:
    raise RuntimeError('No non-empty record set found.')

df = dataframes[selected_record_set_id]

print(f"Working with RecordSet @id: {selected_record_set_id}\nColumns: {df.columns.tolist()}")

# Try to auto-detect a numeric field (float/int columns)
numeric_field_id = None
for col in df.columns:
    # Try to convert, skip non-numeric
    try:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
        df[col] = pd.to_numeric(df[col], errors='raise')
        numeric_field_id = col
        break
    except Exception:
        continue

if numeric_field_id is None:
    raise ValueError('No numeric field found in record set '+selected_record_set_id)
    
print(f"Selected numeric field @id for filtering/normalization: {numeric_field_id}")

# Apply some filtering
threshold = df[numeric_field_id].mean()  # Example: above mean
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered {len(filtered_df)} records in '{selected_record_set_id}' with '{numeric_field_id}' > {threshold:.2f}")

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() or 1)
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Auto-detect a categorical field to group by (first non-numeric)
group_field_id = None
for col in df.columns:
    if col == numeric_field_id: continue
    # Heuristic: dtype object, not many unique values
    if df[col].dtype == object and df[col].nunique() < len(df) // 2:
        group_field_id = col
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}' (@id):")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields, referencing with their `@id`.

We use matplotlib for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(6, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of '{numeric_field_id}' in Record Set '{selected_record_set_id}'")
plt.xlabel(numeric_field_id)
plt.show()

if group_field_id:
    # Boxplot grouped by the categorical field
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"'{numeric_field_id}' grouped by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. We discovered available record sets and fields, extracted data by referencing their `@id`s, and performed basic filtering, normalization, grouping, and plotting on selected columns.

This workflow demonstrates a FAIR and reproducible approach to complex, well-described scientific data. For deeper domain-specific analysis, continue by examining the specific semantic definitions of fields and applying relevant statistical models.